# FinAgent Retrieval 평가 (Google Colab)

validation 80문항에서 `300/50`, `400/60`, `500/80`을 비교합니다.

- 후보 확보: BM25 / Dense / Hybrid의 Recall@20
- 최종 정렬: Hybrid / Hybrid+Reranker의 MRR@5, nDCG@5
- Dense: `BAAI/bge-m3`
- Reranker: `BAAI/bge-reranker-v2-m3`

먼저 Colab 메뉴에서 **런타임 → 런타임 유형 변경 → T4 GPU**를 선택하세요. 비공개 저장소 접근을 위해 Colab Secrets에 `GITHUB_TOKEN`을 등록하고 Notebook access를 켜야 합니다. `KUBIG_FINANCE` 저장소를 `/content/KUBIG_FINANCE`에 clone하며, cache와 결과만 Google Drive에 보존합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
from pathlib import Path
from google.colab import userdata
import base64
import os
import shutil

REPO_URL = 'https://github.com/youhan200203/KUBIG_FINANCE.git'
BRANCH = 'codex/retrieval-eval-validation'
PROJECT_DIR = Path('/content/KUBIG_FINANCE')
DRIVE_OUTPUT_DIR = Path('/content/drive/MyDrive/FinAgent_retrieval_eval')
DRIVE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

github_token = userdata.get('GITHUB_TOKEN')
if not github_token:
    raise RuntimeError('Colab Secrets에 GITHUB_TOKEN을 등록하고 Notebook access를 켜세요.')

credential = base64.b64encode(
    f'x-access-token:{github_token}'.encode('utf-8')
).decode('ascii')
GIT_AUTH_HEADER = f'Authorization: Basic {credential}'

if (PROJECT_DIR / '.git').is_dir():
    print('이미 clone된 저장소를 사용합니다:', PROJECT_DIR)
elif PROJECT_DIR.exists():
    raise RuntimeError(f'{PROJECT_DIR}가 존재하지만 Git 저장소가 아닙니다.')
else:
    !git -c http.extraHeader="{GIT_AUTH_HEADER}" clone --branch "{BRANCH}" --single-branch "{REPO_URL}" "{PROJECT_DIR}"
    if not (PROJECT_DIR / '.git').is_dir():
        raise RuntimeError('Git clone 실패: 토큰 권한, 저장소 선택, 브랜치 이름을 확인하세요.')
    print(f'clone 완료: {REPO_URL} ({BRANCH})')

EVAL_DIR = PROJECT_DIR / 'retrieval_eval'
DATASET_PATH = PROJECT_DIR / 'rag_evaluation_dataset.jsonl'
REQUIREMENTS_PATH = str(EVAL_DIR / 'requirements.txt')
PREP_SCRIPT = str(EVAL_DIR / 'prepare_retrieval_data.py')
EVAL_SCRIPT = str(EVAL_DIR / 'eval_retrieval.py')

required = [
    EVAL_DIR / 'eval_retrieval.py',
    EVAL_DIR / 'prepare_retrieval_data.py',
    EVAL_DIR / 'requirements.txt',
    DATASET_PATH,
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(
        'PROJECT_DIR가 잘못되었거나 필요한 파일이 없습니다:\n' + '\n'.join(missing)
    )

DRIVE_CACHE_DIR = DRIVE_OUTPUT_DIR / 'cache'
DRIVE_CACHE_DIR.mkdir(parents=True, exist_ok=True)
LOCAL_CACHE_DIR = EVAL_DIR / 'cache'
if LOCAL_CACHE_DIR.exists() and not LOCAL_CACHE_DIR.is_symlink():
    shutil.copytree(LOCAL_CACHE_DIR, DRIVE_CACHE_DIR, dirs_exist_ok=True)
    shutil.rmtree(LOCAL_CACHE_DIR)
if not LOCAL_CACHE_DIR.exists():
    LOCAL_CACHE_DIR.symlink_to(DRIVE_CACHE_DIR, target_is_directory=True)

os.chdir(PROJECT_DIR)
print('PROJECT_DIR:', PROJECT_DIR)
print('DATASET:', DATASET_PATH)
print('DRIVE OUTPUT:', DRIVE_OUTPUT_DIR)

In [ ]:
%pip install -q -r {REQUIREMENTS_PATH}
print('의존성 설치 완료')

In [ ]:
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError(
        'GPU가 활성화되지 않았습니다. 런타임 → 런타임 유형 변경 → T4 GPU를 선택하세요.'
    )
print('GPU:', torch.cuda.get_device_name(0))
!nvidia-smi

## 데이터셋 검증

PDF/Web 문서 수, validation 80/test 40 분할, 청크 ID와 gold ID, PDF 숫자 셀 보존 여부를 검사합니다.

In [ ]:
%cd {PROJECT_DIR}
!python -u {PREP_SCRIPT} --check-only

## Validation 평가 실행

세 청크 설정을 동일한 validation 80문항으로 평가합니다. Dense embedding은 `retrieval_eval/cache`에 저장되어 다음 실행부터 재사용됩니다. 처음 실행할 때 모델 다운로드와 embedding 생성 때문에 시간이 걸릴 수 있습니다.

In [ ]:
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
os.environ['PYTHONUNBUFFERED'] = '1'
%cd {PROJECT_DIR}
!python -u {EVAL_SCRIPT} --all-chunk-sizes --split validation --candidate-k 20 --diagnostic-k 50 --final-k 5

## 결과 확인

완료되면 청크 설정별 JSON 세 개와 통합 Markdown 보고서를 Google Drive의 `MyDrive/FinAgent_retrieval_eval`에 복사합니다.

In [ ]:
import json
from IPython.display import Markdown, display

REPORT_PATH = EVAL_DIR / 'reports' / 'initial_method_chunk_comparison.md'
RESULT_PATHS = [
    EVAL_DIR / 'results' / f'results_{variant}_ko_validation.json'
    for variant in ('300_50', '400_60', '500_80')
]

for path in RESULT_PATHS:
    if not path.exists():
        raise FileNotFoundError(f'평가 결과가 없습니다: {path}')
    result = json.loads(path.read_text(encoding='utf-8'))
    print(path.name, '| chunks:', result['n_chunks'], '| questions:', result['n_questions'])
    shutil.copy2(path, DRIVE_OUTPUT_DIR / path.name)

shutil.copy2(REPORT_PATH, DRIVE_OUTPUT_DIR / REPORT_PATH.name)
display(Markdown(REPORT_PATH.read_text(encoding='utf-8')))
print('통합 보고서:', DRIVE_OUTPUT_DIR / REPORT_PATH.name)